In [1]:
import os

os.environ["NEURON_EXPLAINER_API_KEY"] = "EMPTY"
os.environ["NEURON_EXPLAINER_API_BASE"] = "http://localhost:8000/v1" # Paste here your vLLM API base URL, with /v1 at the end

In [2]:
from neuron_explainer.activations.activation_records import calculate_max_activation
from neuron_explainer.activations.activations import ActivationRecordSliceParams, load_neuron
from neuron_explainer.explanations.calibrated_simulator import UncalibratedNeuronSimulator
from neuron_explainer.explanations.explainer import TokenActivationPairExplainer
from neuron_explainer.explanations.prompt_builder import PromptFormat
from neuron_explainer.explanations.scoring import simulate_and_score
from neuron_explainer.explanations.simulator import ExplanationNeuronSimulator, ExplanationTokenByTokenSimulator

In [3]:
EXPLAINER_MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
SIMULATOR_MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

In [4]:
from neuron_explainer.api_client import ApiClient

In [5]:
client = ApiClient(model_name="Qwen/Qwen3-Coder-30B-A3B-Instruct", max_concurrent=1)

In [6]:
test_response = await client.make_request(messages=[{"role": "user", "content": "What's up?"}], max_tokens=2)
print("Response:", test_response["choices"][0]["message"])

Response: {'role': 'assistant', 'content': 'Nothing much', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning_content': None}


In [7]:
best_score = -float('inf')
i = 0
while best_score < 0.3 and i < 6000:
    # Load a neuron record.
    neuron_record = load_neuron(0, i)

    # Grab the activation records we'll need.
    slice_params = ActivationRecordSliceParams(n_examples_per_split=5)
    train_activation_records = neuron_record.train_activation_records(
        activation_record_slice_params=slice_params
    )
    valid_activation_records = neuron_record.valid_activation_records(
        activation_record_slice_params=slice_params
    )

    # Generate an explanation for the neuron.
    explainer = TokenActivationPairExplainer(
        model_name=EXPLAINER_MODEL_NAME,
        prompt_format=PromptFormat.HARMONY_V4,
        max_concurrent=1,
    )

    explanations = await explainer.generate_explanations(
        all_activation_records=train_activation_records,
        max_activation=calculate_max_activation(train_activation_records),
        num_samples=1,
    )
    assert len(explanations) == 1
    explanation = explanations[0]
    # print(f"{explanation=}")

    # Simulate and score the explanation.
    simulator = UncalibratedNeuronSimulator(
        ExplanationNeuronSimulator(
            SIMULATOR_MODEL_NAME,
            explanation,
            max_concurrent=1,
            prompt_format=PromptFormat.INSTRUCTION_FOLLOWING,
        )
    )
    scored_simulation = await simulate_and_score(simulator, valid_activation_records)
    print(f"score={scored_simulation.get_preferred_score():.2f}")
    best_score = max(best_score, scored_simulation.get_preferred_score())
    i += 1
print(i-1, best_score)

score=0.29
score=0.17
score=0.10
score=-0.01
score=0.02
score=0.17
score=0.15
score=0.16
score=0.19
score=0.33
9 0.3267773440844178


In [8]:
neuron_record = load_neuron(0, 9)

In [9]:
slice_params = ActivationRecordSliceParams(n_examples_per_split=5)
train_activation_records = neuron_record.train_activation_records(
    activation_record_slice_params=slice_params
)
valid_activation_records = neuron_record.valid_activation_records(
    activation_record_slice_params=slice_params
)

In [10]:
# Generate an explanation for the neuron.
explainer = TokenActivationPairExplainer(
    model_name=EXPLAINER_MODEL_NAME,
    prompt_format=PromptFormat.HARMONY_V4,
    max_concurrent=1,
)

explanations = await explainer.generate_explanations(
    all_activation_records=train_activation_records,
    max_activation=calculate_max_activation(train_activation_records),
    num_samples=1,
)
assert len(explanations) == 1
explanation = explanations[0]
print(f"{explanation=}")

explanation=' words related to legal restrictions and prohibitions.'


In [14]:
def print_simulations(simulations):
    for s in simulations:
        for t, ea, ta in zip(
            s.simulation.tokens, s.simulation.expected_activations, s.true_activations
        ):
            print(f"{t}\t{ea:.4f}\t{ta:.4f}")


In [13]:
# Simulate and score the explanation.
simulator = UncalibratedNeuronSimulator(
    ExplanationNeuronSimulator(
        SIMULATOR_MODEL_NAME,
        explanation,
        max_concurrent=1,
        prompt_format=PromptFormat.INSTRUCTION_FOLLOWING,
    )
)
scored_simulation = await simulate_and_score(simulator, valid_activation_records)
print(f"score={scored_simulation.get_preferred_score():.2f}")

score=0.31


In [15]:
print_simulations(
    scored_simulation.scored_sequence_simulations[:3]
)

Both	0.0196	-0.1630
 Francois	0.0913	0.1606
 Beau	1.4928	-0.1598
che	0.3191	-0.1486
min	0.2504	-0.1635
 and	0.0045	-0.0992
 Erik	0.0237	0.0703
 Johnson	0.3166	0.0987
 currently	0.0171	-0.1597
 have	0.0098	-0.1538
 no	0.2080	-0.1506
-	0.0100	-0.1633
move	0.2622	-0.0466
ment	0.1078	0.2410
 clauses	0.1191	3.3867
 (	0.0112	-0.1101
N	0.1157	-0.1630
MC	0.7724	-0.1581
)	0.0990	-0.0089
 involved	0.0153	-0.1492
 in	0.0062	-0.1615
 their	0.0175	-0.1157
 contracts	0.0285	0.6060
.	0.0063	-0.1588
 So	0.0039	-0.1346
,	0.0038	-0.1362
 both	0.0378	-0.1385
 of	0.0277	-0.1492
 them	0.0731	-0.1153
 must	0.0387	-0.1605
 be	0.0612	-0.1486
 protected	0.4144	1.6006
 because	0.0215	-0.1285
 any	0.0648	-0.1583
 player	0.0318	-0.0863
 unwilling	0.0347	0.2527
 to	0.0280	-0.1390
 change	0.1005	0.2004
 their	0.0588	-0.0833
 N	0.4038	-0.1621
MC	0.4693	-0.1628
 before	0.0404	-0.1575
 the	0.0238	-0.0894
 expansion	0.0882	-0.1626
 draft	0.1906	0.0190
 must	0.0630	-0.1598
 be	0.1869	-0.1533
 protected	1.1889	1.4434
.	0